![Redis](https://redis.io/wp-content/uploads/2024/04/Logotype.svg?auto=webp&quality=85,75&width=120)

# Featureform Transformations & Feature Lineage

In this recipe we build features with **SQL transformations** in [**Featureform**](https://docs.featureform.com/), and follow the **lineage** Featureform records from raw data all the way to a served feature.

## Why transformations and lineage matter
A production feature is rarely a raw column — it's the result of transformations that clean and aggregate the raw events. Two things go wrong without a feature store:
- **Nobody can tell how a value was produced.** When a model misbehaves you need to trace a feature back to its source. Ad-hoc scripts don't record that path.
- **Steps get duplicated and drift.** The "clean transactions" logic gets rewritten slightly differently by every downstream job.

Featureform fixes both by making every transformation a **named, versioned resource** whose input is another named resource. That dependency graph *is* the lineage: `raw source → transformation → feature`, recorded and visualizable, with each step defined exactly once.

## What we'll build
Two SQL transformations over a transactions dataset, each derived from the same raw source:
1. **`clean_transactions`** — filters out invalid rows from the raw source.
2. **`avg_user_transaction`** — aggregates each user's valid transactions into an average and a count.

Each transformation is a named node whose input is the `transactions` source, so Featureform records a lineage edge `transactions → <transformation>` for each. We then define an `avg_transaction_amt` feature from `avg_user_transaction`, a label from `clean_transactions`, register a training set, `apply()` everything, and walk each node — seeing the exact DataFrame Featureform computed.

## The stack — no Spark, no cloud

Featureform separates **compute/offline** (where transformations run) from the **online store** (where features are served). Transformations here are **SQL**, which run directly in a SQL offline store — so there's no Spark cluster and no object storage to stand up. This recipe uses:
- **ClickHouse** as the offline store — a columnar SQL database that runs the transformations. One local Docker container.
- **Redis** as the online store, for low-latency serving of the finished feature.

> ℹ️ **Transformations are SQL, not pandas here.** Featureform's pandas (`df_transformation`) support requires a Spark or Kubernetes provider. SQL transformations cover the same clean → aggregate → serve pipeline with none of that infrastructure.

> ⚠️ **This notebook needs local Docker and will not run on Colab or in CI.** The cells below start all three pieces for you:
> 1. A **ClickHouse** container (offline store).
> 2. A **Redis** container (online store).
> 3. The **Featureform** coordinator (gRPC on `localhost:7878`, dashboard on `http://localhost`) — started with `featureform deploy docker` after the pip install.

### Start ClickHouse and Redis

This notebook launches its own containers on a private Docker network (`ff-net`) so the Featureform coordinator can reach them **by container name** — no reliance on host ports, which other processes may already be using. The containers are named `ff-clickhouse` / `ff-redis` (project-scoped, so they won't collide with anything else on your machine). Only ClickHouse's HTTP port is published to the host — on `18123` — so the cells below can load data; the native port and Redis stay inside `ff-net`, reached only by the coordinator.

In [1]:
# NBVAL_SKIP
# Launch this notebook's own containers on a private Docker network (ff-net) so the
# Featureform coordinator reaches them by name — no host-port collisions. Names are
# project-scoped (ff-clickhouse/ff-redis) so they won't clash with other containers on
# your machine. Only ClickHouse's HTTP port is published (on 18123, to avoid the common
# 8123) so this notebook can load data; the native port and Redis stay inside ff-net.
# ClickHouse is pinned to 24.10 — the native-protocol version the coordinator speaks.
!docker rm -f ff-clickhouse ff-redis 2>/dev/null
!docker network create ff-net 2>/dev/null || true
!docker run -d --network ff-net --name ff-clickhouse -p 18123:8123 -e CLICKHOUSE_PASSWORD=featureform clickhouse/clickhouse-server:24.10
!docker run -d --network ff-net --name ff-redis redis:8

b499f2dd3bdcf1cb2211f7e82a7713f61a95b2ee25e24f541af1a607f5b6f783
0c6f537968b1eeaf1414c60de80d3f2981efd10288e4302c7c07011e5af88b39
5dd6a2ed98bd9c7519dfe4705eb2985d722a00b8649ecada48def33d693b4a60


## Environment Setup

### Install Python Dependencies

In [2]:
%pip install -q featureform redis clickhouse-connect pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
# NBVAL_SKIP
import sys
# Start the Featureform coordinator (gRPC on localhost:7878, dashboard on http://localhost).
# Invoke via the kernel's own interpreter so it works even if the `featureform` console
# script isn't on PATH (common in Jupyter/VSCode after %pip install). Then attach the
# coordinator to ff-net, retrying until confirmed, so it resolves ClickHouse/Redis by name.
!{sys.executable} -m featureform deploy docker
!for i in $(seq 10); do docker network connect ff-net featureform 2>/dev/null; docker inspect featureform --format '{{range $k,$v := .NetworkSettings.Networks}}{{$k}} {{end}}' 2>/dev/null | grep -q ff-net && echo "coordinator attached to ff-net" && break; sleep 1; done

Deploying Featureform on Docker
Starting Docker deployment on Darwin 24.6.0
Checking if featureform container exists...
	Container featureform not found. Creating new container...
	'featureform' container started

Featureform is now running!
To access the dashboard, visit http://localhost:80
To apply definition files, run `featureform apply <file.py> --host http://localhost:7878 --insecure`
coordinator attached to ff-net


### Configure connections

Connection values are driven by environment variables so you can point the notebook at your own instances. The defaults are the **container names** on the shared `ff-net` network (`clickhouse`, `redis`) — how the coordinator container resolves the providers. (Data is loaded into ClickHouse from this notebook over the published HTTP port `8123` on `localhost`, in the cell further below.)

In [4]:
import os

# Featureform coordinator (gRPC)
FEATUREFORM_HOST = os.getenv("FEATUREFORM_HOST", "localhost:7878")

# The coordinator reaches the providers by container name over the shared ff-net network.
# ClickHouse offline store (recent images require a password for network access)
CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "ff-clickhouse")
CLICKHOUSE_NATIVE_PORT = int(os.getenv("CLICKHOUSE_NATIVE_PORT", "9000"))
CLICKHOUSE_HTTP_PORT = int(os.getenv("CLICKHOUSE_HTTP_PORT", "18123"))  # published to host for the data-load cell
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "default")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "featureform")
CLICKHOUSE_DATABASE = os.getenv("CLICKHOUSE_DATABASE", "default")

# Redis online store
REDIS_HOST = os.getenv("REDIS_HOST", "ff-redis")
REDIS_PORT = int(os.getenv("REDIS_PORT", "6379"))
REDIS_PASSWORD = os.getenv("REDIS_PASSWORD", "")

### Create a sample table in ClickHouse

So the notebook is self-contained, we create a `transactions` table and load a small, intentionally *messy* dataset (some invalid negative amounts) so the first transformation has something to clean. In a real deployment this table would already exist. We connect over ClickHouse's published HTTP port `18123` from here.

In [5]:
# NBVAL_SKIP
import time
import clickhouse_connect
import numpy as np

# ClickHouse may still be starting up — retry until it accepts connections.
for _ in range(30):
    try:
        ch = clickhouse_connect.get_client(
            host="localhost", port=CLICKHOUSE_HTTP_PORT,
            username=CLICKHOUSE_USER, password=CLICKHOUSE_PASSWORD,
        )
        ch.command("SELECT 1")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ClickHouse not reachable on localhost:8123 — is the container running?")

ch.command("DROP TABLE IF EXISTS transactions")
ch.command(
    """
    CREATE TABLE transactions (
        TransactionID String,
        CustomerID String,
        TransactionAmount Float64,
        IsFraud Bool
    ) ENGINE = MergeTree ORDER BY CustomerID
    """
)

rng = np.random.default_rng(42)
n = 500
rows = []
for i in range(n):
    amount = round(float(rng.gamma(2.0, 50.0)), 2)
    if i % 150 == 0:
        amount = -1.0   # a few invalid amounts for the cleaning step to drop
    rows.append([
        f"T{i:05d}",
        f"C{int(rng.integers(1000, 1050)):04d}",
        amount,
        bool(rng.integers(0, 2)),
    ])

ch.insert("transactions", rows,
          column_names=["TransactionID", "CustomerID", "TransactionAmount", "IsFraud"])
print("rows in ClickHouse:", ch.command("SELECT count() FROM transactions"))

rows in ClickHouse: 500


## Register the providers

We register the **ClickHouse** offline store and the **Redis** online store. Registering a provider just tells Featureform how to reach it — no data moves yet.

In [6]:
import featureform as ff

clickhouse = ff.register_clickhouse(
    name="clickhouse-quickstart",
    description="ClickHouse offline store (runs the SQL transformations)",
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_NATIVE_PORT,
    user=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

redis = ff.register_redis(
    name="redis-quickstart",
    description="Redis online (inference) store",
    host=REDIS_HOST,
    port=REDIS_PORT,
    password=REDIS_PASSWORD,
    db=0,
)

## Register the raw source

We point Featureform at the ClickHouse `transactions` table. This becomes the **root node** of our lineage graph — the source that transformations build on.

In [7]:
transactions = clickhouse.register_table(
    name="transactions",
    variant="quickstart",
    table="transactions",  # the table name in ClickHouse
)

## Step 1 — a cleaning transformation

A `sql_transformation` is a plain function that returns a SQL string. The `{{transactions.quickstart}}` placeholder references the source we just registered. This query runs **in ClickHouse**, and its result becomes a new, named source. Here we drop the invalid (non-positive) amounts.

In [8]:
@clickhouse.sql_transformation(variant="quickstart")
def clean_transactions():
    """Keep only valid transactions."""
    return (
        "SELECT CustomerID, TransactionAmount, IsFraud "
        "FROM {{transactions.quickstart}} WHERE TransactionAmount > 0"
    )

## Step 2 — a second transformation off the same source

`avg_user_transaction` reads the **same `{{transactions.quickstart}}` source** and aggregates each user's valid transactions into an average and a count. It's a *sibling* of `clean_transactions`, not chained onto it: both are named, versioned nodes rooted in the raw source, so Featureform records a lineage edge from `transactions` to each.

> ℹ️ Chaining one transformation directly onto another's output (`FROM {{clean_transactions.quickstart}}`) requires a Spark or Snowflake offline store. On a SQL offline store like ClickHouse, transformations read registered sources, so both transformations here derive from `transactions`.

In [9]:
@clickhouse.sql_transformation(variant="quickstart")
def avg_user_transaction():
    """Average transaction amount and count per user, over the valid transactions."""
    return (
        "SELECT CustomerID AS user_id, "
        "avg(TransactionAmount) AS avg_transaction_amt, "
        "count(*) AS transaction_count "
        "FROM {{transactions.quickstart}} WHERE TransactionAmount > 0 GROUP BY CustomerID"
    )

## Define the entity, feature, and label

`@ff.entity` groups resources keyed by a **user**. The feature is sourced from the `avg_user_transaction` transformation and materialized to Redis for serving. The label (`IsFraud`) comes from the `clean_transactions` transformation and stays offline — it's only used to build training sets.

Feature and label draw from **two different transformations rooted in the same raw source**, so they share a common lineage back to `transactions`.

In [10]:
@ff.entity
class User:
    avg_transactions = ff.Feature(
        avg_user_transaction[["user_id", "avg_transaction_amt"]],
        variant="quickstart",
        type=ff.Float32,
        inference_store=redis,
    )
    fraudulent = ff.Label(
        clean_transactions[["CustomerID", "IsFraud"]],
        variant="quickstart",
        type=ff.Bool,
    )

## Register a training set

A training set joins the feature(s) to the label on the entity key — built from the same definitions that serve online, so there's no training-serving skew.

In [11]:
ff.register_training_set(
    "fraud_training",
    variant="quickstart",
    label=("fraudulent", "quickstart"),
    features=[("avg_transactions", "quickstart")],
)

TrainingSetVariant(name='fraud_training', owner='default_owner', label=('fraudulent', 'quickstart'), features=[('avg_transactions', 'quickstart')], description='', variant='quickstart', feature_lags=[], tags=[], properties={}, created=None, schedule='', schedule_obj=None, provider='', status='NO_STATUS', error=None, server_status=None, resource_snowflake_config=None, type=<TrainingSetType.DYNAMIC: 1>)

## Apply the definitions

`client.apply()` sends everything to the coordinator and runs the pipeline: ClickHouse executes both transformations against the `transactions` source, then the feature is materialized into Redis. `asynchronous=False` blocks until it finishes.

In [12]:
# NBVAL_SKIP
client = ff.Client(host=FEATUREFORM_HOST, insecure=True)
client.apply(asynchronous=False, verbose=True)

Applying Run: 2026-07-24t15-39-09
Creating User default_owner 
Creating Provider clickhouse-quickstart 
Creating Provider redis-quickstart 
Creating Source Variant transactions quickstart
Creating Source Variant clean_transactions quickstart
Creating Source Variant avg_user_transaction quickstart
Creating Entity user 
Creating Feature Variant avg_transactions quickstart
Creating Label Variant fraudulent quickstart
Creating Trainingset Variant fraud_training quickstart



UserWarning: install "ipywidgets" for Jupyter support

## Follow the lineage stage by stage

`client.dataframe()` computes and returns the DataFrame at any node in the graph. Reading them — the raw source, then each transformation derived from it — you can *see* every node that feeds the served feature. This is the lineage, made concrete.

In [13]:
# NBVAL_SKIP
print("1. Raw source:")
display(client.dataframe(transactions).head())

print("2. After clean_transactions (invalid amounts gone):")
display(client.dataframe(clean_transactions).head())

print("3. After avg_user_transaction (aggregated per user):")
display(client.dataframe(avg_user_transaction).head())

1. Raw source:
No resources to apply


,TransactionID,CustomerID,TransactionAmount,IsFraud
0,T00067,C1000,211.20,True
1,T00093,C1000,110.80,False
2,T00295,C1000,80.36,False
3,T00485,C1000,103.49,False
4,T00062,C1001,26.65,False


2. After clean_transactions (invalid amounts gone):
No resources to apply


,CustomerID,TransactionAmount,IsFraud
0,C1000,211.20,True
1,C1000,110.80,False
2,C1000,80.36,False
3,C1000,103.49,False
4,C1001,26.65,False


3. After avg_user_transaction (aggregated per user):
No resources to apply


,user_id,avg_transaction_amt,transaction_count
0,C1047,118.431111,9
1,C1032,173.796667,6
2,C1006,118.632857,7
3,C1017,85.532500,8
4,C1023,116.270769,13


### See the lineage graph visually

The Featureform dashboard at **http://localhost** renders the dependency DAG for every resource. Open the `avg_transactions` feature and you'll see its lineage `transactions → avg_user_transaction → avg_transactions`; the `clean_transactions` transformation (which feeds the label) hangs off the same `transactions` source — the sibling structure you just walked in code, plus variants, owners, and timestamps for audit.

## Serve the finished feature from Redis

The end of the pipeline: request the feature for a single entity key. This read is served from Redis at low latency — the value having flowed through the whole traceable pipeline to get here. We grab a `user_id` that appears in the aggregated output above.

In [14]:
# NBVAL_SKIP
user_id = client.dataframe(avg_user_transaction)["user_id"].iloc[0]
avg_txn = client.features(
    [("avg_transactions", "quickstart")],
    {"user": user_id},
)
print(f"avg_transactions for user {user_id}:", avg_txn)

No resources to apply
avg_transactions for user C1047: [118.43111419677734]


## Build a training set from the same definitions

The offline side reuses the exact feature/label definitions. The dataset is iterable and streams rows of `(features, label)`.

In [15]:
# NBVAL_SKIP
dataset = client.training_set("fraud_training", "quickstart")

for i, row in enumerate(dataset):
    print(row.features(), "->", row.label())
    if i >= 4:
        break

[[126.4625]] -> [ True]
[[126.4625]] -> [False]
[[126.4625]] -> [False]
[[126.4625]] -> [False]
[[66.58714286]] -> [False]


## Cleanup

Stop and remove the containers when you're done.

In [16]:
# NBVAL_SKIP
import sys
# Tear down everything this notebook started. Remove containers (including the coordinator)
# before the network; the coordinator's endpoint releases asynchronously, so retry the delete.
!{sys.executable} -m featureform stop docker
!docker rm -f ff-clickhouse ff-redis featureform 2>/dev/null
!for i in $(seq 5); do docker network rm ff-net 2>/dev/null && break || sleep 1; done

Tearing down Featureform on Docker
Stopping containers...
	Stopping featureform container
Container quickstart-clickhouse not found. Skipping...
ff-clickhouse
ff-redis
featureform
ff-net


## Learn more

- [Featureform transformations](https://docs.featureform.com/) — SQL and DataFrame transformations, chaining, and variants
- [Featureform + Redis fraud detection recipe](./02_featureform_fraud_detection.ipynb) — a single-transformation version of this pattern